In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier
#from mysql.connector import connect
from sklearn import impute
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, make_scorer, fbeta_score, accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime

In [ ]:
#read/load the data
df = pd.read_excel('structed_features_2020_12_31_public.xlsx',engine='openpyxl') # for public companies
#df = pd.read_excel('structed_features_2020_12_31_private.xlsx',engine='openpyxl') # for private companies
df = df.rename(columns={name: name.lower() for name in df})
df = df[:]

In [ ]:
df

In [ ]:
df['y'] = df['isbad'].astype(int)

In [ ]:
feature_names = [col for col in df if col.startswith('m') or col.startswith('f')]
print(feature_names)

In [ ]:
# data pre-processing
df['m007'][np.isnan(df['m007'])] = 0
df['m009'][np.isnan(df['m009'])] = 0
df['m011'][np.isnan(df['m011'])] = 0 # only for public dataset
df['m012'][np.isnan(df['m012'])] = 0
df['m013'][np.isnan(df['m013'])] = 0
df['m016'][np.isnan(df['m016'])] = 0

df['m014'][np.isnan(df['m014'])] = 1
df['m015'][np.isnan(df['m015'])] = 1

df['m008'][np.isnan(df['m008'])] = 90
df['m010'][np.isnan(df['m010'])] = 90

In [ ]:
simple_imputer = impute.SimpleImputer(strategy='mean')
X = np.array(df[feature_names])
X = simple_imputer.fit_transform(X)

In [ ]:
# Convert the NumPy array to a DataFrame
check = pd.DataFrame(X)

# Save the DataFrame to a CSV file
check.to_csv('check.csv', index=True)

In [ ]:
import pickle
#save final imputer models
model_path = 'imputer1.pkl'
#model_path = 'imputer2.pkl'
with open(model_path, 'wb')  as f:
    pickle.dump(simple_imputer, f)

In [ ]:
df

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=3)

In [ ]:
#dataset info
print('training samples: %s' %len(train_df))
print('test samples: %s' %len(test_df))
print('training positive samples: %s, percentage %s' %(train_df['y'].sum(), train_df['y'].sum()/len(train_df)))
print('test positive samples: %s, percentage %s' %(test_df['y'].sum(), test_df['y'].sum()/len(test_df)))

In [ ]:
X_train = np.array(train_df[feature_names])
X_test = np.array(test_df[feature_names])

In [ ]:
X_train

In [ ]:
simple_imputer = impute.SimpleImputer(strategy='mean')
X_train = simple_imputer.fit_transform(X_train)
X_test = simple_imputer.transform(X_test)

In [ ]:
X_train

In [ ]:
bool_train_labels = np.array(train_df.pop('y'))
y_train = bool_train_labels.astype(int)
bool_test_labels = np.array(test_df.pop('y'))
y_test = bool_test_labels.astype(int)

In [ ]:
def _get_score(estimator, X, y):
    y_pred = estimator.predict(X)
    post_label_idx = np.where(estimator.classes_==1)[0][0]
    y_pred_proba = estimator.predict_proba(X)[:, pos_label_idx]
    return recall_score(y, y_pred), precision_score(y, y_pred), f1_score(y, y_pred), roc_auc_score(y, y_pred_proba), accuracy_score(y, y_pred)

In [ ]:
import pickle
with open(r'model1.pkl', 'rb') as f:
#with open(r'model2.pkl', 'rb') as f:
    clf = pickle.load(f)

In [ ]:
X_test

In [ ]:
pos_label_idx = clf.classes_.tolist().index(1)
y_test_pred = clf.predict(X_test)
y_test_pred_prob = clf.predict_proba(X_test)[:, pos_label_idx]

In [ ]:
scores = _get_score(clf, X_test, y_test)

In [ ]:
scores

In [ ]:
import csv
with open(r'sample_prediction_combined_public.csv', 'w', encoding='utf-8-sig') as f1:
    csv_write = csv.writer(f1)
    title_row = ['ent_name', 'pred_prob']
    csv_write.writerow(title_row)
    
#start data mining process, publish probability of all bust companies
for i in range(len(X_test)):
    if y_test[i]==1 and y_test_pred[i]==1:
        print(test_df[i:i+1]['name'])
        print(y_test_pred_prob[i])
        df_pred = pd.DataFrame({'ent_name': test_df[i:i+1]['name'], 'pred_prob': y_test_pred_prob[i]})
        df_pred.to_csv('sample_prediction_combined_public.csv', mode='a', header=False, index=False)
        